# Ray checkpointing example

This notebook runs a **Ray Train** checkpointing demo on **Red Hat OpenShift AI** using the CodeFlare SDK:

- Submit a **RayJob** with a **managed Ray cluster** (`ClusterConfiguration`) so KubeRay lifecycles the cluster with the job (`shutdownAfterJobFinishes`).
- Configure **S3 access** via an OpenShift AI **Data Connection** attached to this workbench (see below).
- **Monitor** training in the Ray dashboard (**Jobs** tab), then **delete and resubmit** the RayJob (`job.delete()` / `job.submit()`) to verify training **resumes from S3** after a simulated interruption. UI Pause/Resume does not reliably stop the Ray driver.

For quota/preemption with **Red Hat build of Kueue**, see the companion blog and OpenShift *AI workloads* documentation.

Training script: `train_with_checkpoints.py` in this directory (same source as the CodeFlare SDK guided demo).

## Import required libraries

In [ ]:
import os
import subprocess
import time

from codeflare_sdk import (
    Codeflare,
    SDKConfig,
    RayJob,
    ClusterConfiguration,
    get_cluster,
)
from codeflare_sdk.common.utils.constants import CUDA_PY312_RUNTIME_IMAGE
from codeflare_sdk.common.utils.utils import get_ray_image_for_python_version
from kube_authkit import AuthConfig


def resolve_ray_image() -> str:
    """Pick a MODH Ray image; fall back to py312 when workbench Python has no default."""
    if os.environ.get("RAY_IMAGE"):
        return os.environ["RAY_IMAGE"]
    image = get_ray_image_for_python_version()
    return image or CUDA_PY312_RUNTIME_IMAGE

## Authenticate to your OpenShift cluster

`Codeflare` is the single entrypoint: it authenticates with kube-authkit and binds the
resulting Kubernetes client to this instance, so objects created here keep using it even
if another `Codeflare` is created later in the same notebook.

In [ ]:
NAMESPACE = "your-namespace"  # Data Science Project where the RayJob runs
JOB_NAME = "checkpointing-job"

# Authenticate to your Kubernetes/OpenShift cluster using kube-authkit

# Option 1: Auto-detect credentials (kubeconfig or in-cluster service account)
# NOTE: In RHOAI Workbenches, auto-detect uses the workbench SA, which cannot
# create RayJobs. Use Option 2 (token). Project edit/admin is enough
# (kuberay-edit-role aggregates); do not grant admin to the workbench SA.
# auth_config = AuthConfig(method="auto")

# Option 2 (Recommended for RHOAI Workbenches): Token-based authentication
# Get your token with: oc whoami -t  (or from the OpenShift console → Copy login command)
auth_config = AuthConfig(
    method="openshift",
    k8s_api_host="https://api.example.com:6443",
    token="sha256~XXXXX",  # oc whoami -t
)

# Option 3: OIDC authentication (for BYOIDC-enabled clusters)
# auth_config = AuthConfig(
#     method="oidc",
#     k8s_api_host="https://api.example.com:6443",
#     oidc_issuer="https://your-oidc-provider.com",
#     client_id="your-client-id",
#     use_device_flow=True,  # Interactive device flow for notebook environments
# )

cf = Codeflare(config=SDKConfig(auth=auth_config, namespace=NAMESPACE))

# Set to False for self-signed / dev API certificates (optional).
cf.client.configuration.verify_ssl = False


def get_odh_training_jobs_url(namespace: str) -> str:
    """OpenShift AI Training jobs page for Delete in the UI."""
    try:
        base = subprocess.check_output(
            ["oc", "get", "consolelink", "rhodslink", "-o", "jsonpath={.spec.href}"],
            text=True,
        ).strip().rstrip("/")
        return f"{base}/develop-train/training-jobs/{namespace}"
    except Exception:
        return f"<your-openshift-ai-dashboard>/develop-train/training-jobs/{namespace}"



print(f"Namespace: {NAMESPACE!r}, RayJob name: {JOB_NAME!r}")

## Configure S3 credentials

Create an **S3 Data Connection** in your OpenShift AI project and attach it to this workbench before running the notebook. Credential keys are injected as environment variables and copied into the RayJob `runtime_env` below.

For temporary credentials (SSO/federated), set `AWS_SESSION_TOKEN` in the notebook environment before running the next cell — the Data Connection UI does not collect session tokens. If the session token expires before you resubmit, resume from S3 fails (`HeadBucket` HTTP 400).

In [ ]:
AWS_ENV_KEYS = (
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "AWS_DEFAULT_REGION",
    "AWS_S3_BUCKET",
    "AWS_SESSION_TOKEN",  # optional; required for temporary credentials
)
AWS_CREDENTIALS = {k: os.environ[k] for k in AWS_ENV_KEYS if os.environ.get(k)}

required = ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_S3_BUCKET")
missing = [k for k in required if k not in AWS_CREDENTIALS]
if missing:
    raise ValueError(
        f"Missing AWS env vars: {missing}. Attach an S3 Data Connection to this workbench "
        "(Settings → Environment variables) or export the variables manually."
    )

if "AWS_DEFAULT_REGION" not in AWS_CREDENTIALS:
    AWS_CREDENTIALS["AWS_DEFAULT_REGION"] = "us-east-1"

# Change this if you see a Ray version mismatch when resuming from S3.
CHECKPOINT_PREFIX = os.environ.get("RAY_CHECKPOINT_PREFIX", "mnist-demo")

print(f"Using bucket: {AWS_CREDENTIALS['AWS_S3_BUCKET']}")
print(f"S3 checkpoint prefix: ray-checkpoints/{CHECKPOINT_PREFIX}")

## Submit RayJob (managed Ray cluster)

In [ ]:
ray_image = resolve_ray_image()
print(f"Using Ray image: {ray_image}")

managed = ClusterConfiguration(
    num_workers=2,
    head_cpu_requests=2,
    head_cpu_limits=4,
    head_memory_requests=4,
    head_memory_limits=8,
    worker_cpu_requests=2,
    worker_cpu_limits=4,
    worker_memory_requests=4,
    worker_memory_limits=8,
    image=ray_image,
    verify_tls=False,
)

job = RayJob(
    job_name=JOB_NAME,
    entrypoint="python train_with_checkpoints.py",
    cluster_config=managed,
    namespace=NAMESPACE,
    api_client=cf.client,
    runtime_env={
        "working_dir": "./",
        "pip": ["torch", "torchvision", "s3fs", "pyarrow"],
        "env_vars": {
            **AWS_CREDENTIALS,
            "RAY_CHECKPOINT_PREFIX": CHECKPOINT_PREFIX,
            "RAY_TRAIN_WORKER_GROUP_START_TIMEOUT_S": "120",  # Allow time for worker scheduling
        },
    },
)

job.submit()
print(job.status())
print(f"RayCluster template name: {job.cluster_name}")
print("Watch logs for: NO CHECKPOINT FOUND - Starting fresh")

## Monitor job progress (status + Ray dashboard Jobs)

Poll `job.status()`, then open the **Ray dashboard** from OpenShift AI (Ray cluster heading on the RayJob page). Do not wait on `cluster.cluster_dashboard_uri()` — on modular-arch clusters the HTTPRoute is in the applications namespace and the SDK probe can hang. Use **Jobs** in the dashboard for live driver logs.

In [ ]:
print(job.status())
print(f"OpenShift AI — Training jobs: {get_odh_training_jobs_url(NAMESPACE)}")
print(f"Find RayJob '{JOB_NAME}' and open the Ray dashboard from the Ray cluster heading.")
print("In the Ray dashboard, open Jobs and stream logs for the training driver.")
print(
    "Wait for at least one full epoch and a checkpoint to S3 before deleting and resubmitting."
)

## Delete RayJob (checkpoint demo)

After logs show at least **one epoch** and a checkpoint written to **S3**, delete the RayJob. Checkpoints remain in S3.

Use **Delete** in the OpenShift AI UI (Training jobs URL printed above), or run the next cell (`job.delete()`). UI Pause/Resume does not reliably stop the Ray driver.

In [ ]:
print("=" * 60)
print("DELETING RayJob (checkpoint demo — checkpoints remain in S3)")
print(f"OpenShift AI — Training jobs: {get_odh_training_jobs_url(NAMESPACE)}")
print("=" * 60)

job.delete()
print("RayJob deleted. Resubmit with the same parameters in the next cell.")

## Resubmit RayJob

Submit again with the **same** `job_name`, `cluster_config`, `entrypoint`, `runtime_env`, and `RAY_CHECKPOINT_PREFIX`. When the RayCluster is back, confirm in the dashboard **Jobs** view: `RESUMING FROM CHECKPOINT - Starting at epoch N`.

In [ ]:
print("=" * 60)
print("RESUBMITTING RayJob after delete")
print("Watch for RESUMING FROM CHECKPOINT in dashboard Jobs logs")
print("=" * 60)

job = RayJob(
    job_name=JOB_NAME,
    entrypoint="python train_with_checkpoints.py",
    cluster_config=managed,
    namespace=NAMESPACE,
    api_client=cf.client,
    runtime_env={
        "working_dir": "./",
        "pip": ["torch", "torchvision", "s3fs", "pyarrow"],
        "env_vars": {
            **AWS_CREDENTIALS,
            "RAY_CHECKPOINT_PREFIX": CHECKPOINT_PREFIX,
            "RAY_TRAIN_WORKER_GROUP_START_TIMEOUT_S": "120",
        },
    },
)
job.submit()
time.sleep(10)
print(job.status())

## Verify resume from checkpoint

In the Ray dashboard **Jobs** tab, look for:

```
RESUMING FROM CHECKPOINT - Starting at epoch N
Previous loss: X.XXXX
```

That confirms optimizer and progress were restored from S3 across the delete/resubmit cycle.

In [ ]:
print(job.status())
print(f"OpenShift AI — Training jobs: {get_odh_training_jobs_url(NAMESPACE)}")
print("Open the Ray dashboard from the Ray cluster heading in OpenShift AI.")
print("Check Jobs tab for: RESUMING FROM CHECKPOINT - Starting at epoch N")

## Cleanup

Use **Delete** in the OpenShift AI UI or execute the next cell to delete the RayJob and tear down the RayCluster if it is still present.

In [ ]:
print("Cleaning up...")
try:
    status_data = job._api.get_job_status(name=job.name, k8s_namespace=NAMESPACE)
    ray_cluster_name = (status_data or {}).get("rayClusterName")
except Exception:
    ray_cluster_name = None

try:
    job.delete()
except Exception:
    pass

if ray_cluster_name:
    try:
        c = get_cluster(
            ray_cluster_name,
            namespace=NAMESPACE,
            verify_tls=False,
            api_client=cf.client,
        )
        if c is not None:
            c.down()
    except Exception:
        pass

print("Cleanup attempted (RayJob delete; cluster.down if RayCluster still exists).")

In [ ]:
# No explicit logout needed - authentication is managed automatically by kube-authkit